# M2.S2 — Shared-memory computing with OpenMP
## Interactive HPC notebook

This notebook accompanies **M2.S2 — Shared-memory computing with OpenMP**.

The lecture moves from a parallel design to safe multicore code. The notebook makes that progression executable:

1. create a team of threads;
2. split loop iterations across threads;
3. reason about shared and private data;
4. observe a race condition;
5. repair it with the right OpenMP construct;
6. compare static and dynamic scheduling;
7. connect OpenMP threads to CPU cores allocated by Slurm.

### Classroom method
> **PREDICT → RUN → OBSERVE → EXPLAIN**

### Recommended environment
- **IE/SciTech JupyterHub + HPC cluster**
- Python 3 kernel
- GCC with OpenMP support (`-fopenmp`)
- Slurm for the final cluster activity

The notebook does **not** install compilers or packages. If GCC/OpenMP is unavailable, the relevant cells will explain what to do.

## Before you start

For every activity, make a prediction before running the code.

The goal is not to memorize pragmas. The goal is to understand the relationship:

**parallel design → OpenMP construct → correct result → performance**

# 0 — Environment and compiler check

### Predict
- Is GCC available?
- Does this GCC support OpenMP?
- Is Slurm visible from this Jupyter environment?

In [ ]:
import os
import platform
import shutil
import subprocess
import textwrap

print('Host:', platform.node())
print('User:', os.environ.get('USER', 'unknown'))
print('Visible CPUs:', os.cpu_count())
print('gcc:', shutil.which('gcc'))
print('sinfo:', shutil.which('sinfo'))
print('sbatch:', shutil.which('sbatch'))

if shutil.which('gcc'):
    p = subprocess.run(['gcc', '--version'], capture_output=True, text=True)
    print('\n', p.stdout.splitlines()[0])

    test = subprocess.run(
        ['bash', '-lc', "printf '#include <omp.h>\\nint main(){return 0;}\\n' | gcc -x c -fopenmp - -o /tmp/omp_test"],
        capture_output=True, text=True
    )
    print('OpenMP compile test:', 'OK' if test.returncode == 0 else 'FAILED')
    if test.returncode != 0:
        print(test.stderr)
else:
    print('GCC is not available. Run the C/OpenMP activities on the SciTech HPC environment.')

In [ ]:
def compile_c(source_file, exe_file, extra_flags=None):
    if shutil.which('gcc') is None:
        print('gcc not found — run this activity on the SciTech HPC environment.')
        return False
    flags = ['gcc', '-fopenmp', '-O2', source_file, '-o', exe_file]
    if extra_flags:
        flags.extend(extra_flags)
    p = subprocess.run(flags, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stderr)
        return False
    return True

def run_exe(exe_file, threads=None, args=None):
    env = os.environ.copy()
    if threads is not None:
        env['OMP_NUM_THREADS'] = str(threads)
    cmd = ['./' + exe_file] + (args or [])
    p = subprocess.run(cmd, capture_output=True, text=True, env=env)
    print(p.stdout, end='')
    if p.stderr:
        print(p.stderr)
    return p

### Observe
A successful OpenMP compile test means GCC accepts `-fopenmp` and can include `omp.h`.

### Explain
Why is seeing many CPUs in `os.cpu_count()` **not** the same thing as having those CPUs allocated to your job?

# 1 — Fork–join: one thread becomes a team
### Slide connection: `#pragma omp parallel`

A parallel region creates a team of threads. Every thread executes the block.

### Predict
1. With `OMP_NUM_THREADS=4`, how many lines should appear?
2. Will the lines necessarily appear in the order 0, 1, 2, 3?
3. What happens after the parallel region ends?

In [ ]:
hello_src = r'''
#include <omp.h>
#include <stdio.h>

int main(void) {
    #pragma omp parallel
    {
        printf("Hello from thread %d of %d\n",
               omp_get_thread_num(), omp_get_num_threads());
    }
    return 0;
}
'''

with open('omp_hello.c', 'w') as f:
    f.write(hello_src)

if compile_c('omp_hello.c', 'omp_hello'):
    print('--- 1 thread ---')
    run_exe('omp_hello', 1)
    print('\n--- 4 threads ---')
    run_exe('omp_hello', 4)

### Observe
- The parallel region executes once per thread.
- Thread identifiers are unique within the team.
- Print order can vary between runs.

### Try it
Run the 4-thread case several times. Do you always get the same order?

### Explain
OpenMP creates concurrency, but why does it not guarantee the order in which threads reach `printf()`?

# 2 — Worksharing: who executes each loop iteration?
### Slide connection: `#pragma omp parallel for`

The algorithm is unchanged: every iteration still runs exactly once. OpenMP distributes independent iterations among threads.

### Predict
For 16 iterations and 4 threads with static scheduling, which thread do you expect to own each group of iterations?

In [ ]:
workshare_src = r'''
#include <omp.h>
#include <stdio.h>

int main(void) {
    const int N = 16;
    int owner[16];

    #pragma omp parallel for schedule(static)
    for (int i = 0; i < N; ++i) {
        owner[i] = omp_get_thread_num();
    }

    for (int i = 0; i < N; ++i)
        printf("iteration %2d -> thread %d\n", i, owner[i]);
    return 0;
}
'''

with open('omp_workshare.c', 'w') as f:
    f.write(workshare_src)

if compile_c('omp_workshare.c', 'omp_workshare'):
    run_exe('omp_workshare', 4)

### Observe
Each iteration appears exactly once, but different threads own different iterations.

### Try it
Change the run to 2 threads, then 8 threads.

### Explain
What changed: the algorithm, the number of iterations, or the assignment of iterations to workers?

# 3 — Shared or private? Decide who owns each variable
### Slide connection: data environment

OpenMP shared-memory programming is easy to start because threads can see the same address space. That is also what makes races possible.

Consider:

- `x`: one value read by all threads;
- `y`: a temporary computed independently by each thread;
- `tid`: the thread identifier.

### Predict
Which should be shared and which should be private/local?

In [ ]:
scope_src = r'''
#include <omp.h>
#include <stdio.h>

int main(void) {
    int x = 10;

    #pragma omp parallel default(none) shared(x)
    {
        int tid = omp_get_thread_num();
        int y = x + tid;
        printf("Thread %d: x=%d, y=%d\n", tid, x, y);
    }
    return 0;
}
'''

with open('omp_scope.c', 'w') as f:
    f.write(scope_src)

if compile_c('omp_scope.c', 'omp_scope'):
    run_exe('omp_scope', 4)

### Observe
- Every thread sees the same `x = 10`.
- Each thread computes its own local `y`.

### Explain
Why is `default(none)` useful while learning OpenMP?

What could go wrong if a temporary variable that is written by every thread were accidentally shared?

# 4 — Race condition: correct serial code can become wrong in parallel
### Slide connection: read → modify → write is not one indivisible operation

We will increment one shared counter many times.

### Predict
With 2,000,000 loop iterations, should the final answer be exactly 2,000,000?

What happens if several threads execute `counter++` at the same time?

In [ ]:
race_src = r'''
#include <omp.h>
#include <stdio.h>

int main(void) {
    const int N = 2000000;

    for (int trial = 0; trial < 5; ++trial) {
        long long counter = 0;

        #pragma omp parallel for
        for (int i = 0; i < N; ++i) {
            counter++;
        }

        printf("trial %d: counter=%lld (expected %d)\n",
               trial + 1, counter, N);
    }
    return 0;
}
'''

with open('omp_race.c', 'w') as f:
    f.write(race_src)

if shutil.which('gcc'):
    p = subprocess.run(['gcc', '-fopenmp', '-O0', 'omp_race.c', '-o', 'omp_race'],
                       capture_output=True, text=True)
    if p.returncode == 0:
        run_exe('omp_race', 8)
    else:
        print(p.stderr)

### Observe
The result may vary between trials and can be smaller than the mathematically correct result.

### Explain
- Why does `counter++` contain a race even though it looks like one C statement?
- Would adding a barrier **after** the loop repair lost updates?

**Key idea:** fast wrong answers are still wrong.

# 5 — Repair the race: atomic, critical, or reduction?
### Slide connection: correct first, then fast

There are several synchronization tools, but they are not equally appropriate.

For an accumulation such as a sum, OpenMP `reduction` is usually the natural pattern.

### Predict
Which approach should scale better for a large sum?

A. one `critical` section around every update  
B. one `atomic` update for every iteration  
C. private partial sums combined with `reduction`

In [ ]:
reduction_src = r'''
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    const int N = 5000000;
    double *v = malloc((size_t)N * sizeof(double));
    if (!v) return 1;

    for (int i = 0; i < N; ++i)
        v[i] = 0.5 + (i % 1000) * 0.001;

    double serial = 0.0;
    double t0 = omp_get_wtime();
    for (int i = 0; i < N; ++i)
        serial += 0.5 * v[i] * v[i];
    double serial_time = omp_get_wtime() - t0;

    double parallel = 0.0;
    t0 = omp_get_wtime();
    #pragma omp parallel for reduction(+:parallel)
    for (int i = 0; i < N; ++i)
        parallel += 0.5 * v[i] * v[i];
    double parallel_time = omp_get_wtime() - t0;

    printf("threads=%d serial=%.12f parallel=%.12f diff=%.3e serial_t=%.6f parallel_t=%.6f\n",
           omp_get_max_threads(), serial, parallel,
           serial - parallel, serial_time, parallel_time);

    free(v);
    return 0;
}
'''

with open('omp_reduction.c', 'w') as f:
    f.write(reduction_src)

if compile_c('omp_reduction.c', 'omp_reduction'):
    for t in [1, 2, 4, 8]:
        run_exe('omp_reduction', t)

### Observe
- The reduction result should match the serial baseline to normal floating-point tolerance.
- Timing may improve as threads increase, but 8 threads do **not** guarantee exactly 8× speedup.

### Explain
Why is `reduction` better suited to this pattern than serializing every single update inside one critical region?

# 6 — Scheduling: static or dynamic?
### Slide connection: keep workers useful

Static scheduling is predictable and efficient when iterations cost about the same. Dynamic scheduling can help when iteration costs vary.

The next program contains irregular work: every fifth iteration is much slower.

### Predict
Which schedule will finish sooner: `static,1` or `dynamic,1`?

In [ ]:
schedule_src = r'''
#include <omp.h>
#include <stdio.h>
#include <unistd.h>

static void run_static(void) {
    const int N = 20;
    int owner[20];
    double t0 = omp_get_wtime();

    #pragma omp parallel for schedule(static,1)
    for (int i = 0; i < N; ++i) {
        owner[i] = omp_get_thread_num();
        usleep((i % 5 == 0) ? 60000 : 6000);
    }

    printf("static  time=%.3f s owners:", omp_get_wtime() - t0);
    for (int i = 0; i < N; ++i) printf(" %d", owner[i]);
    printf("\n");
}

static void run_dynamic(void) {
    const int N = 20;
    int owner[20];
    double t0 = omp_get_wtime();

    #pragma omp parallel for schedule(dynamic,1)
    for (int i = 0; i < N; ++i) {
        owner[i] = omp_get_thread_num();
        usleep((i % 5 == 0) ? 60000 : 6000);
    }

    printf("dynamic time=%.3f s owners:", omp_get_wtime() - t0);
    for (int i = 0; i < N; ++i) printf(" %d", owner[i]);
    printf("\n");
}

int main(void) {
    run_static();
    run_dynamic();
    return 0;
}
'''

with open('omp_schedule.c', 'w') as f:
    f.write(schedule_src)

if compile_c('omp_schedule.c', 'omp_schedule'):
    run_exe('omp_schedule', 4)

### Observe
- Static assignment is regular and predictable.
- Dynamic assignment changes which thread receives later iterations.
- For irregular work, dynamic scheduling may reduce idle time.

### Try it
Change the slow-iteration condition from `i % 5 == 0` to `i % 2 == 0` and compare again.

### Explain
Scheduling changes **who executes the work**, not what the algorithm computes.

# 7 — Slurm gives you cores; OpenMP uses them
### Slide connection: allocation and thread creation are different decisions

A typical shared-memory job requests CPU cores from Slurm and then tells OpenMP how many threads to create.

### Predict
If Slurm allocates 4 CPUs but `OMP_NUM_THREADS=8`, what problem might you create?

In [ ]:
if shutil.which('sinfo') is None:
    print('Slurm commands are not available here. Run this section on the SciTech HPC environment.')
else:
    print('--- available Slurm resources (summary) ---')
    subprocess.run(['sinfo', '-h', '-o', '%P %a %l %D %c'], check=False)

In [ ]:
slurm_script = '''#!/bin/bash
#SBATCH --job-name=m2s2_omp
#SBATCH --cpus-per-task=4
#SBATCH --time=00:01:00
#SBATCH --output=m2s2_omp-%j.out

export OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK
echo "host=$(hostname)"
echo "SLURM_CPUS_PER_TASK=$SLURM_CPUS_PER_TASK"
echo "OMP_NUM_THREADS=$OMP_NUM_THREADS"
srun ./omp_hello
'''

with open('m2s2_openmp.slurm', 'w') as f:
    f.write(slurm_script)

print(slurm_script)

### Run on the cluster
If your SciTech account has a default partition/account, submit from a terminal with:

```bash
sbatch m2s2_openmp.slurm
squeue -u $USER
```

When the job finishes:

```bash
cat m2s2_omp-<jobid>.out
```

If SciTech requires a partition/account, add the values specified by SciTech. Do not guess them.

### Explain
- Slurm decides which CPU resources your job may use.
- OpenMP decides how many threads your program creates.
- Why should those two choices normally agree?

# Challenge — Parallelize it safely

You are given this serial pattern:

```c
double total = 0.0;
for (int i = 0; i < N; ++i) {
    double x = transform(input[i]);
    output[i] = x;
    total += x;
}
```

In pairs, answer before writing any code:

1. Which loop work is independent?
2. Which variables are shared?
3. Which values should be private?
4. Why would a naïve `parallel for` create a race?
5. Which OpenMP clause would you use for `total`?
6. Which scheduling strategy would you start with if every iteration costs about the same?
7. What Slurm resource would you request for 8 OpenMP threads?

Then write the OpenMP version and compare its output with the serial baseline.

# What did we learn?

1. **`parallel` creates a team of threads.**
2. **`parallel for` distributes independent loop iterations across the team.**
3. **Shared memory makes cooperation easy and races possible.**
4. **Variable ownership matters: shared, private/local, and reduction are different patterns.**
5. **Correctness comes first; reduction and synchronization repair specific hazards.**
6. **Scheduling, overhead and load balance affect performance even when code is correct.**
7. **Slurm allocates CPU cores; OpenMP creates threads to use them.**

## Next
In **M2.S3 — MPI**, we move beyond one shared-memory node. When memory is no longer shared, processes must exchange data explicitly.